In [1]:
from elasticsearch import Elasticsearch
import numpy as np

In [2]:
# 准备数据
items = [
    {"type": "phone", "id": "用户A", "number": 13800001234},
    {"type": "phone", "id": "用户B", "number": 13800005678},
    {"type": "order", "id": "订单1001", "number": 203011010001},
    {"type": "order", "id": "订单1002", "number": 203011010123},
    {"type": "order", "id": "订单2001", "number": 203012150045},
    {"type": "phone", "id": "用户C", "number": 13912345678},
    {"type": "phone", "id": "用户D", "number": 13798765432},
    {"type": "order", "id": "订单3001", "number": 205001020333},
    {"type": "order", "id": "订单3002", "number": 205001020777},
    {"type": "phone", "id": "用户E", "number": 13622223333},
]

In [3]:
# 链接ES
from numpy import indices


es = Elasticsearch(
    hosts = [{'host':'localhost','port':9200,'scheme':'http'}],
    request_timeout = 30 # 默认0秒会触发超时
)

# 创建索引参数
index_name = 'number_vector'
mapping = {
    "mappings": {
        "properties": {
            "number_vector": {
                "type": "dense_vector",
                "dims": 1,
                "index": True,
                "similarity": "l2_norm"  # L2距离
            },
            "type":{"type":"keyword"},
            "id":{"type":"keyword"},
           "number":{"type":"long"}
        }
    }
}

# 删除已存在的索引
if es.indices.exists(index=index_name):  # 索引存在
    es.indices.delete(index=index_name)

# 创建索引
try:
    es.indices.create(index=index_name, mappings=mapping['mappings'])
except TypeError:
    es.indices.create(index=index_name, body=mapping)

In [4]:
# 构建数据插入数据库
dimension = 1

for i ,item in enumerate(items):
    number_vector = [float(item['number'])]
    es.index(
        index = index_name,
        id = i+1,
        document= {
            "number_vector": number_vector,
            "type": item['type'],
            "id": item['id'],
            "number": item['number']
        }
    )
print(f'已向索引添加 {len(items)} 个数字向量 (维度={dimension})')
# 刷新索引
es.indices.refresh(index=index_name)

已向索引添加 10 个数字向量 (维度=1)


ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

In [ ]:
query_number = 205001020500
query_vector = [float(query_number)]

k = 5 # 搜索最接近的5个向量

knn_query ={
    'size':k,
    '_source':['type','id','number','number_vector'], #搜索结果返回的字段
    'knn':{
        'field':'number_vector',
        'query_vector':query_vector,
        'k':k,
        'num_candidates':100  # 粗排阶段候选向量数量,越大精度消耗的资源越多
    }
}

results = es.search(index=index_name,body=knn_query)

print("\n查询数字：", query_number)
print(f"Top-{k} 最相近数字（按 L2 距离，越小越相近）：")

print(results)


查询数字： 205001020500
Top-5 最相近数字（按 L2 距离，越小越相近）：
{'took': 143, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 5, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'number_vector', '_id': '1', '_score': 1.0, '_source': {'number': 13800001234, 'number_vector': [13800002000.0], 'id': '用户A', 'type': 'phone'}}, {'_index': 'number_vector', '_id': '8', '_score': 1.0, '_source': {'number': 205001020333, 'number_vector': [205001020000.0], 'id': '订单3001', 'type': 'order'}}, {'_index': 'number_vector', '_id': '9', '_score': 1.0, '_source': {'number': 205001020777, 'number_vector': [205001020000.0], 'id': '订单3002', 'type': 'order'}}, {'_index': 'number_vector', '_id': '3', '_score': 4.411857e-19, '_source': {'number': 203011010001, 'number_vector': [203011010000.0], 'id': '订单1001', 'type': 'order'}}, {'_index': 'number_vector', '_id': '4', '_score': 4.411857e-19, '_source': {'number': 203011010123, 'number_vector': [203011

In [6]:
for rank, hit in enumerate(results["hits"]["hits"], start=1):
    source = hit['_source']

    print(f"{rank}. 得分={hit['_score']:.4f} | 类型={source['type']} | 标识={source['id']} | 数字={source['number']}")
    

1. 得分=1.0000 | 类型=phone | 标识=用户A | 数字=13800001234
2. 得分=1.0000 | 类型=order | 标识=订单3001 | 数字=205001020333
3. 得分=1.0000 | 类型=order | 标识=订单3002 | 数字=205001020777
4. 得分=0.0000 | 类型=order | 标识=订单1001 | 数字=203011010001
5. 得分=0.0000 | 类型=order | 标识=订单1002 | 数字=203011010123
